### This notebook fits a reference spline to HF and AB reference data

In [2]:
import os
import sys

sys.path.append(os.path.abspath("../../../"))

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import os
from glob2 import glob
from pathlib import Path
from src.functions.plot_functions import format_2d_plotly


/Users/nick/Projects/repositories/morphseq/src/functions/__init__.py:23: UserWarning: Optional legacy import failed: src.functions.core_utils_segmentation: No module named 'segmentation_models_pytorch'
  warnings.warn(f"Optional legacy import failed: {module}: {e}")
/Users/nick/Projects/repositories/morphseq/src/functions/__init__.py:23: UserWarning: Optional legacy import failed: src.functions.spline_morph_spline_metrics: No module named 'seaborn'
  warnings.warn(f"Optional legacy import failed: {module}: {e}")
/Users/nick/Projects/repositories/morphseq/src/functions/__init__.py:23: UserWarning: Optional legacy import failed: src.functions.embryo_df_performance_metrics: No module named 'seaborn'
  warnings.warn(f"Optional legacy import failed: {module}: {e}")


In [3]:
# load embryo_df for our current best model
# root = "/media/nick/hdd02/Cole Trapnell's Lab Dropbox/Nick Lammers/Nick/morphseq/"

root = "/Users/nick/Cole Trapnell's Lab Dropbox/Nick Lammers/Nick/morphseq/"
train_name = "20241107_ds"
model_name = "SeqVAE_z100_ne150_sweep_01_block01_iter030" 
train_dir = Path(root) / "training_data" / train_name
output_dir = train_dir / model_name

# get path to model
training_path = sorted(glob(str(output_dir / "*")))[-1]
training_name = Path(training_path).name
read_path = Path(training_path) / "figures" 

# path to save data
out_path = Path(root) / "results" / "20260504"
os.makedirs(out_path, exist_ok=True)

# path to figures and data
fig_path = Path(root) / "slides" / "morphseq" / "20260504" / "morph_metrics"
os.makedirs(fig_path, exist_ok=True)

In [4]:
morph_df = pd.read_csv(read_path / "embryo_stats_df.csv", index_col=0)
umap_df = pd.read_csv(read_path / "umap_df.csv", index_col=0)
print(umap_df.shape)
umap_df = umap_df.merge(morph_df.loc[:, ["snip_id", "embryo_id", "experiment_time"]], how="left", on=["snip_id"])
print(umap_df.shape)

/var/folders/m7/tpjttxb92tl8b4c9svgvgby00000gn/T/ipykernel_51985/2326850292.py:1: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  morph_df = pd.read_csv(read_path / "embryo_stats_df.csv", index_col=0)


(96210, 22)
(96210, 24)


/var/folders/m7/tpjttxb92tl8b4c9svgvgby00000gn/T/ipykernel_51985/2326850292.py:2: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  umap_df = pd.read_csv(read_path / "umap_df.csv", index_col=0)


### Make 3D UMAP and PCA for hotfish experiments

In [5]:
HF_experiments = np.asarray(['20240813_24hpf', '20240813_30hpf', '20240813_36hpf']) #, '20240813_extras'])
hf_morph_df = morph_df.loc[np.isin(morph_df["experiment_date"], HF_experiments), :].reset_index()
hf_umap_df = umap_df.loc[np.isin(umap_df["experiment_date"], HF_experiments), :].reset_index()

hf_outlier_snips = np.asarray(["20240813_24hpf_F06_e00_t0000", "20240813_36hpf_D03_e00_t0000", "20240813_36hpf_C03_e00_t0000"]) 
hf_umap_df = hf_umap_df.loc[~np.isin(hf_umap_df["snip_id"], hf_outlier_snips), :]
hf_morph_df = hf_morph_df.loc[~np.isin(hf_morph_df["snip_id"], hf_outlier_snips), :]

## Calculate pairwise correlation between latent morph features 

In [6]:
col_prefix = "z_mu_b"

In [7]:
hf_morph_df["temperature"].unique()

array([22.])

In [8]:
# Load the curated Hotfish morph PCA metadata to recover the correct embryo temperatures.
morph_latent_read_path = Path(root) / "results" / "20250312" / "morph_latent_space"
hf_pca_morph_df = pd.read_csv(morph_latent_read_path / "hf_pca_morph_df.csv")

hf_temp_meta_df = hf_pca_morph_df.loc[:, ["snip_id", "temperature", "timepoint"]].drop_duplicates("snip_id")

# Keep the latent morphology columns from hf_morph_df, but replace the stale 22C metadata with the Hotfish metadata.
hf_morph_df = hf_morph_df.drop(columns=[col for col in ["temperature", "timepoint"] if col in hf_morph_df.columns])
hf_morph_df = hf_morph_df.merge(hf_temp_meta_df, how="left", on="snip_id", validate="one_to_one")

missing_temp = hf_morph_df["temperature"].isna().sum()
if missing_temp > 0:
    raise ValueError(f"Missing Hotfish temperature metadata for {missing_temp} snips")

hf_morph_df.loc[:, ["snip_id", "temperature", "timepoint"]].head()


,snip_id,temperature,timepoint
0,20240813_24hpf_A02_e00_t0000,19.0,24
1,20240813_24hpf_A03_e00_t0000,25.0,24
2,20240813_24hpf_A04_e00_t0000,28.5,24
3,20240813_24hpf_A05_e00_t0000,32.0,24
4,20240813_24hpf_A06_e00_t0000,33.5,24


In [9]:
from tqdm import tqdm
import plotly.colors as pc
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage, dendrogram

n_emb_thresh = 8
feature_cols = [col for col in hf_morph_df.columns if col.startswith(col_prefix)]
feature_cols = sorted(feature_cols, key=lambda x: int(x.split("_")[-1]) if x.split("_")[-1].isdigit() else x)

if len(feature_cols) == 0:
    raise ValueError(f"No columns found with prefix '{col_prefix}'")

feature_cc_fig_path = fig_path / f"{col_prefix}_feature_cc"
feature_cc_fig_path.mkdir(parents=True, exist_ok=True)

# calculate correlation coefficients by embryo temperature
feature_temp_vec = np.asarray(sorted(hf_morph_df["temperature"].dropna().unique())).astype(float)
feature_cc_mat_list = []
feature_cc_mat_list_null = []
feature_cov_mat_list = []
feature_count_mat_list = []

np.random.seed(371)

for temp in tqdm(feature_temp_vec):
    temp_df = hf_morph_df.loc[hf_morph_df["temperature"] == temp, feature_cols]
    temp_df = temp_df.apply(pd.to_numeric, errors="coerce")

    correlation_matrix = temp_df.corr(method="pearson")
    covariance_matrix = temp_df.cov()

    bool_df = temp_df.notna()
    count_matrix = bool_df.astype(int).T.dot(bool_df.astype(int))
    feature_count_mat_list.append(count_matrix.to_numpy())

    feature_cc_mat_list.append(correlation_matrix)
    feature_cov_mat_list.append(covariance_matrix)

    vals_raw = temp_df.values.flatten()
    vals = vals_raw[~np.isnan(vals_raw)]
    shuffled_values = np.random.choice(vals, len(vals_raw), replace=True).reshape(temp_df.shape)
    null_df = pd.DataFrame(shuffled_values, index=temp_df.index, columns=temp_df.columns)

    null_correlation_matrix = null_df.corr(method="pearson")
    feature_cc_mat_list_null.append(null_correlation_matrix)


100%|██████████| 6/6 [00:00<00:00, 184.71it/s]


In [10]:
# filter out features with too few paired embryos in any retained temperature group
stacked_counts = np.stack(feature_count_mat_list, axis=2)
min_count_array = np.squeeze(np.min(stacked_counts, axis=2))
feature_filter_vec = np.max(min_count_array, axis=0) >= n_emb_thresh
feature_colnames = np.asarray(feature_cc_mat_list[0].columns)

for t in range(len(feature_temp_vec)):
    # corr
    cc_temp = feature_cc_mat_list[t].copy()
    feature_cc_mat_list[t] = cc_temp.loc[feature_colnames[feature_filter_vec], feature_colnames[feature_filter_vec]]

    # null corr
    cc_temp_n = feature_cc_mat_list_null[t].copy()
    feature_cc_mat_list_null[t] = cc_temp_n.loc[feature_colnames[feature_filter_vec], feature_colnames[feature_filter_vec]]

    # cov
    cv_temp = feature_cov_mat_list[t].copy()
    feature_cov_mat_list[t] = cv_temp.loc[feature_colnames[feature_filter_vec], feature_colnames[feature_filter_vec]]

print(f"Retained {feature_filter_vec.sum()} of {len(feature_filter_vec)} {col_prefix} features")


Retained 80 of 80 z_mu_b features


In [11]:
feature_temp_vec

array([19. , 25. , 28.5, 32. , 33.5, 35. ])

In [12]:
# get sort order from the 28.5C control group when available, otherwise use the middle temperature group
if 28.5 in feature_temp_vec:
    t_ind = int(np.where(feature_temp_vec == 28.5)[0][0])
else:
    t_ind = len(feature_temp_vec) // 2

corr = feature_cc_mat_list[t_ind].copy()
corr_for_cluster = corr.fillna(0)
np.fill_diagonal(corr_for_cluster.values, 1)

# Compute hierarchical clustering on 1 - Pearson correlation.
dist = 1 - corr_for_cluster
condensed_dist = squareform(dist, checks=False)
Z = linkage(condensed_dist, method="average")
dendro = dendrogram(Z, no_plot=True)
feature_order = dendro["leaves"]


In [13]:
feature_temp_vec

array([19. , 25. , 28.5, 32. , 33.5, 35. ])

In [23]:
# for t, temp in enumerate(feature_temp_vec[-1:]):

t = 3
temp = feature_temp_vec[t]
print(temp)

cc_mat = feature_cc_mat_list[t].iloc[feature_order, feature_order]
fig = px.imshow(cc_mat, color_continuous_scale="RdBu_r", range_color=[-1, 1])

fig.update_layout(width=600, height=600)
fig.update_layout(
    xaxis_title="",
    yaxis_title="",
    font=dict(color="white", family="Arial, sans-serif", size=18)
)

fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)
fig.update_traces(xgap=1, ygap=1)
fig.update_layout(plot_bgcolor="black", paper_bgcolor="black")
fig.update_layout(coloraxis_showscale=False)
fig.update_layout(margin=dict(l=10, r=10, t=10, b=10))
fig.update_xaxes(automargin=False)
fig.update_yaxes(automargin=False)

fig.write_image(feature_cc_fig_path / f"{col_prefix}_feature_cc_{temp:g}C.png", scale=2)
fig.write_html(feature_cc_fig_path / f"{col_prefix}_feature_cc_{temp:g}C.html")
cc_mat.to_csv(feature_cc_fig_path / f"{col_prefix}_feature_cc_{temp:g}C.csv", index=True)

fig.show()


32.0


In [15]:
feature_cc_list = []
feature_cc_null_val_list = []

for t, temp in enumerate(feature_temp_vec):
    cc = feature_cc_mat_list[t].to_numpy()
    tril_indices = np.tril_indices(cc.shape[0], k=-1)
    cc_vals = cc[tril_indices]

    cc_n = feature_cc_mat_list_null[t].to_numpy()
    cc_n_vals = cc_n[tril_indices]
    feature_cc_null_val_list.append(cc_n_vals[cc_vals <= 1])

    cc_df_temp = pd.DataFrame(cc_vals[cc_vals <= 1], columns=["cc"])
    cc_df_temp["temperature"] = temp
    feature_cc_list.append(cc_df_temp)

feature_null_df = pd.DataFrame(np.concatenate(feature_cc_null_val_list), columns=["cc"])
feature_null_df["temperature"] = "null"

feature_cc_df = pd.concat(feature_cc_list + [feature_null_df], axis=0, ignore_index=True)
feature_cc_df.head()


,cc,temperature
0,-0.229864,19.0
1,-0.797157,19.0
2,-0.337713,19.0
3,-0.448033,19.0
4,0.963372,19.0


In [17]:
feature_cc_plot_df = feature_cc_df.loc[feature_cc_df["temperature"] != "null"].copy()
feature_cc_plot_df["temperature"] = feature_cc_plot_df["temperature"].astype(float)
feature_cc_plot_df["cc_abs"] = feature_cc_plot_df["cc"].abs()

feature_group_stats = feature_cc_plot_df.groupby(["temperature"])["cc_abs"].agg(
    mean="mean",
    std="std",
    sem="sem"
).reset_index()

groups = feature_cc_plot_df["temperature"].unique()
min_val, max_val = 17, 38
colors = dict({float(val):
    pc.sample_colorscale("RdBu_r", (val - min_val) / (max_val - min_val))[0]
    for val in groups
})

feature_cc_plot_df["temperature_group"] = pd.Categorical(feature_cc_plot_df["temperature"].astype(str))
feature_cc_plot_df = feature_cc_plot_df.sort_values(["temperature_group"])

fig = px.box(feature_cc_plot_df, y="cc_abs", x="temperature", color="temperature", color_discrete_map=colors)
fig.update_traces(width=0.8)
fig = format_2d_plotly(fig, font_size=20, axis_labels=["temperature (C)", "correlation coefficient"])
fig.update_traces(opacity=0.95)
fig.update_traces(line=dict(width=5))

fig.show()

fig.write_image(feature_cc_fig_path / f"{col_prefix}_feature_cc_box.png", scale=2)
fig.write_html(feature_cc_fig_path / f"{col_prefix}_feature_cc_box.html")


In [18]:
fig = px.scatter(
    feature_group_stats,
    x="temperature",
    y="mean",
    error_y="sem",
    color="temperature",
    color_continuous_scale="RdBu_r",
    range_color=[17, 38]
)

fig = format_2d_plotly(fig, axis_labels=["temperature (C)", "correlation coefficient"], font_size=20, marker_size=24)
fig.update_traces(mode="lines+markers", line=dict(color="white"))
# fig.update_layout(yaxis=dict(range=[0, 0.3]))
fig.update_layout(coloraxis_showscale=False)
fig.update_traces(error_x=dict(color="white", width=0))

fig.show()

fig.write_image(feature_cc_fig_path / f"{col_prefix}_feature_cc_scatter.png", scale=2)
fig.write_html(feature_cc_fig_path / f"{col_prefix}_feature_cc_scatter.html")


In [19]:
import numpy as np
import pandas as pd
import plotly.express as px

# -----------------------------
# Metric functions
# -----------------------------

def _offdiag_values(R):
    R = np.asarray(R, dtype=float)
    mask = ~np.eye(R.shape[0], dtype=bool)
    return R[mask]

def mean_abs_offdiag_corr(R):
    vals = _offdiag_values(R)
    return np.nanmean(np.abs(vals))

def rms_offdiag_corr(R):
    vals = _offdiag_values(R)
    return np.sqrt(np.nanmean(vals ** 2))

def effective_rank(R, eps=1e-12):
    R = np.asarray(R, dtype=float)

    # Replace NaNs if needed
    R = np.nan_to_num(R, nan=0.0)

    eigvals = np.linalg.eigvalsh(R)
    eigvals = np.clip(eigvals, 0, None)

    if eigvals.sum() <= eps:
        return np.nan

    q = eigvals / eigvals.sum()
    entropy = -np.sum(q * np.log(q + eps))
    return np.exp(entropy)

def top_k_eigen_fraction(R, k=3, eps=1e-12):
    R = np.asarray(R, dtype=float)
    R = np.nan_to_num(R, nan=0.0)

    eigvals = np.linalg.eigvalsh(R)
    eigvals = np.clip(eigvals, 0, None)
    eigvals = np.sort(eigvals)[::-1]

    if eigvals.sum() <= eps:
        return np.nan

    return eigvals[:k].sum() / eigvals.sum()


In [27]:
# -----------------------------
# Calculate metrics
# -----------------------------

metric_records = []

for t in range(len(feature_cc_mat_list)):
    cc_mat = feature_cc_mat_list[t].iloc[feature_order, feature_order]

    metric_records.append({
        "temperature": float(t),
        "mean_abs_offdiag_corr": mean_abs_offdiag_corr(cc_mat),
        "rms_offdiag_corr": rms_offdiag_corr(cc_mat),
        "effective_rank": effective_rank(cc_mat),
        "top1_eigen_fraction": top_k_eigen_fraction(cc_mat, k=1),
        "top3_eigen_fraction": top_k_eigen_fraction(cc_mat, k=3),
        "top5_eigen_fraction": top_k_eigen_fraction(cc_mat, k=5),
    })

feature_order_metric_df = pd.DataFrame(metric_records).sort_values("temperature")


# -----------------------------
# Plot helper
# -----------------------------

def plot_metric(metric, y_label=None):
    if y_label is None:
        y_label = metric.replace("_", " ")

    fig = px.scatter(
        feature_order_metric_df,
        x="temperature",
        y=metric,
        color="temperature",
        color_continuous_scale="RdBu_r",
        range_color=[17, 38],
    )

    fig = format_2d_plotly(
        fig,
        axis_labels=["temperature (C)", y_label],
        font_size=20,
        marker_size=24,
    )

    fig.update_traces(
        mode="lines+markers",
        line=dict(color="white"),
    )

    fig.update_layout(coloraxis_showscale=False)

    fig.show()

    fig.write_image(feature_cc_fig_path / f"{col_prefix}_{metric}.png", scale=2)
    fig.write_html(feature_cc_fig_path / f"{col_prefix}_{metric}.html")

    return fig


# -----------------------------
# Make plots
# -----------------------------

plot_metric("mean_abs_offdiag_corr", "mean |correlation coefficient|")
plot_metric("rms_offdiag_corr", "RMS off-diagonal correlation")
plot_metric("effective_rank", "effective rank")
plot_metric("top1_eigen_fraction", "top 1 eigenvalue fraction")
plot_metric("top3_eigen_fraction", "top 3 eigenvalue fraction")
plot_metric("top5_eigen_fraction", "top 5 eigenvalue fraction")

In [26]:
feature_cc_mat_list[0].shape

(80, 80)